In [2]:
%pip install openpyxl pillow


Note: you may need to restart the kernel to use updated packages.


In [9]:
import os
import difflib
from openpyxl import load_workbook
from openpyxl.drawing.image import Image as XLImage
from PIL import Image as PILImage
import io
import re

# Ask user for paths
excel_file = input("Enter Excel file path: ").strip().strip('"')
image_folder = input("Enter Images Folder path: ").strip().strip('"')

# Load workbook
wb = load_workbook(excel_file)
ws = wb.active

IMG_WIDTH_PX = 90
IMG_HEIGHT_PX = 90
ROW_HEIGHT_PT = IMG_HEIGHT_PX * 0.75
COL_WIDTH_CHARS = 13

def normalize(name):
    """
    Normalize name for matching:
    - Lowercase
    - Remove spaces between single letters (initials)
    - e.g. "Ali C P" → "ali cp", "Aysha M I" → "aysha mi"
    """
    name = name.lower().strip()
    # Merge consecutive single-letter words (initials) together
    # e.g. "c p" → "cp", "k p" → "kp", "m i" → "mi"
    name = re.sub(r'\b([a-z])\s+([a-z])\b', r'\1\2', name)
    name = re.sub(r'\b([a-z])\s+([a-z])\b', r'\1\2', name)  # run twice for 3 initials
    return name

# Build folder map: normalized name → full path
folder_files = {}
for fname in os.listdir(image_folder):
    name_only, ext = os.path.splitext(fname)
    if ext.lower() in [".jpg", ".jpeg", ".png"]:
        key = normalize(name_only.strip())
        folder_files[key] = os.path.join(image_folder, fname)

print(f"\nFound {len(folder_files)} images in folder.")
print("\nProcessing...\n")

for row in range(2, ws.max_row + 1):

    student_name = ws[f"A{row}"].value
    if student_name is None:
        continue

    student_name = str(student_name).strip()
    student_norm = normalize(student_name)

    matched_path = None
    matched_by = ""

    # ✅ Method 1: Exact match after normalization
    if student_norm in folder_files:
        matched_path = folder_files[student_norm]
        matched_by = "exact"

    # ✅ Method 2: Excel name starts with file name
    # e.g. Excel="muhammed anufas t" → File="muhammed anufas"
    if not matched_path:
        for file_key, file_path in folder_files.items():
            if student_norm.startswith(file_key):
                matched_path = file_path
                matched_by = f"starts-with (file: '{file_key}')"
                break

    # ✅ Method 3: File name starts with Excel name
    if not matched_path:
        for file_key, file_path in folder_files.items():
            if file_key.startswith(student_norm):
                matched_path = file_path
                matched_by = f"file-starts-with (file: '{file_key}')"
                break

    # ✅ Method 4: Fuzzy match using difflib (catches spelling diffs)
    # e.g. "mohammad" vs "mohammed"
    if not matched_path:
        all_keys = list(folder_files.keys())
        close = difflib.get_close_matches(student_norm, all_keys, n=1, cutoff=0.75)
        if close:
            matched_path = folder_files[close[0]]
            matched_by = f"fuzzy (file: '{close[0]}')"

    if matched_path:
        try:
            pil_img = PILImage.open(matched_path)
            pil_img = pil_img.convert("RGB")
            pil_img = pil_img.resize((IMG_WIDTH_PX, IMG_HEIGHT_PX), PILImage.LANCZOS)

            img_buffer = io.BytesIO()
            pil_img.save(img_buffer, format="PNG")
            img_buffer.seek(0)

            xl_img = XLImage(img_buffer)
            xl_img.width = IMG_WIDTH_PX
            xl_img.height = IMG_HEIGHT_PX

            ws.add_image(xl_img, f"W{row}")
            ws.row_dimensions[row].height = ROW_HEIGHT_PT

            print(f"✓ {student_name}  [{matched_by}]")

        except Exception as e:
            print(f"✗ Error for {student_name}: {e}")
    else:
        print(f"✗ No match found: '{student_name}'")

ws.column_dimensions["W"].width = COL_WIDTH_CHARS

output_file = os.path.join(
    os.path.dirname(excel_file),
    "students_with_images.xlsx"
)

wb.save(output_file)

print("\n✅ Done!")
print(f"Output saved at:\n{output_file}")

Enter Excel file path:  "C:\Users\MSU\Desktop\Learner Data_BBA_Batch 1 Sem 1, 2_Done.xlsx
Enter Images Folder path:  C:\Users\MSU\Desktop\Batch 1



Found 35 images in folder.

Processing...

✓ Aboobakar Thalhath C  [exact]
✓ Ahmed Shahin  [fuzzy (file: 'ahamed shahin tp')]
✓ Ali Ashique P V  [starts-with (file: 'ali ashique')]
✓ Aneena Anees  [exact]
✓ Ansif Muhammed A  [starts-with (file: 'ansif muhammed')]
✓ Asma Ummer Zainuddin  [exact]
✓ Fathima Fidha M C  [exact]
✓ Fathima Ibrahimkutty  [exact]
✓ Fathima Nada A  [exact]
✓ Fathima Rebea  [exact]
✓ Fathimath Dilfa Tk  [exact]
✓ Fathimathul Najiya K C  [exact]
✓ Hadiya Nasrin  [exact]
✓ Mohamed Saloob Kalangadan  [fuzzy (file: 'mohamed  saloob kalangadan')]
✓ Mohammed Hafin Ashraf K  [exact]
✓ Mohammed Labeeb Kozhikkattil  [fuzzy (file: 'mohammed labeeb kozhikattil')]
✓ Mohammed Musthafa  [fuzzy (file: 'mohammed  musthafa')]
✓ Muhammad Rizan  [fuzzy (file: 'muhammed rizan')]
✓ Muhammed Aaqil K T  [exact]
✓ Muhammed Gasni  [exact]
✓ Muhammed Hasan  [fuzzy (file: 'muhammed hassan')]
✓ Muhammed Ihithisham  [exact]
✓ Muhammed Nasith.C.K  [starts-with (file: 'muhammed nasith')]
✓ Mu